In [160]:
import pandas as pd
import pyodbc

sheet_curr = '1wxR3iYna86qrdViwHjUPzHuw6bCNeMLb72M25hpUHYk'
sheet_archive = '1PxNYGMXaVrRqI0uyMQF46K7nDEG16WnDoKrFyI_qrvE'
gid_matches = '2141931777'
gid_deck = '590005429'

In [ ]:
def conn(query):
    conn_str = (
        "Driver={ODBC Driver 18 for SQL Server};"
        "Server=mtgo-data-collection.database.windows.net;"
        "Database=mtgo-results;"
        "Uid=X;"
        "Pwd=X;"
    )
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()

    cursor.execute(query)

    conn.commit()
    cursor.close()
    conn.close()

In [152]:
def parse_class_sheet(sheet,gid):
    deck_url = f'https://docs.google.com/spreadsheets/d/{sheet}/export?format=csv&gid={gid}'
    df = pd.read_csv(deck_url)

    # Create dataframe with valid Deck Names.
    df_decks = df[['Archetype','Subarchetype']].sort_values(['Archetype','Subarchetype']).reset_index(drop=True)
    df_decks.columns = ['ARCHETYPE','SUBARCHETYPE']
    df_decks['ARCHETYPE'] = df_decks['ARCHETYPE'].str.strip().str.upper()
    df_decks['SUBARCHETYPE'] = df_decks['SUBARCHETYPE'].str.strip().str.upper()

    # Adding Deck IDs.
    count = 1001
    df_decks['DECK_ID'] = 0
    for index, row in df_decks.iterrows():
        df_decks.at[index,'DECK_ID'] = count
        count += 1

    df_decks = pd.concat([df_decks,pd.DataFrame({'ARCHETYPE':['NA','NA'],'SUBARCHETYPE':['NA','NO SHOW'],'DECK_ID':[1998,1999]})],ignore_index=True)

    # Create dataframe with valid Event Types.
    df_events = df[['Event Types']]
    df_events.columns = ['EVENT_TYPE']
    df_events = df_events.dropna(subset=['EVENT_TYPE'])
    df_events['EVENT_TYPE'] = df_events['EVENT_TYPE'].str.strip().str.upper()

    # Adding Event Type IDs.
    count = 101
    df_events['EVENT_TYPE_ID'] = 0
    for index, row in df_events.iterrows():
        df_events.at[index,'EVENT_TYPE_ID'] = count
        count += 1

    # Add Format column.
    df_decks['FORMAT'] = 'VINTAGE'
    df_decks = df_decks[['FORMAT','ARCHETYPE','SUBARCHETYPE','DECK_ID']]

    df_events['FORMAT'] = 'VINTAGE'
    df_events = df_events[['FORMAT','EVENT_TYPE','EVENT_TYPE_ID']]
    
    return (df_decks,df_events)

In [179]:
def parse_matchup_sheet(sheet,gid):
    sheet_url = f'https://docs.google.com/spreadsheets/d/{sheet}/export?format=csv&gid={gid}'
    df = pd.read_csv(sheet_url)

    # Rename columns.
    df.columns = ['P1','P2','P1_WINS','P2_WINS','WINNER1','WINNER2','P1_ARCH','P2_ARCH','P1_SUBARCH','P2_SUBARCH','P1_NOTE','P2_NOTE','EVENT_DATE','EVENT_TYPE']

    # Replace null values with 'NA' string.
    df.fillna({'P1_ARCH':'NA','P2_ARCH':'NA','P1_SUBARCH':'NA','P2_SUBARCH':'NA','P1_NOTE':'NA','P2_NOTE':'NA'},inplace=True)

    # Strip whitespace from player/deck names.
    df.P1 = df.P1.str.strip()
    df.P2 = df.P2.str.strip()
    df.P1_ARCH = df.P1_ARCH.str.strip().str.upper()
    df.P2_ARCH = df.P2_ARCH.str.strip().str.upper()
    df.P1_SUBARCH = df.P1_SUBARCH.str.strip().str.upper()
    df.P2_SUBARCH = df.P2_SUBARCH.str.strip().str.upper()
    df.P1_NOTE = df.P1_NOTE.str.strip().str.upper()
    df.P2_NOTE = df.P2_NOTE.str.strip().str.upper()

    # Format EVENT_TYPE values.
    df['EVENT_TYPE'] = df['EVENT_TYPE'].str.upper().str.strip()

    # Format No Show deck name values.
    for index, row in df.iterrows():
        if row['P1_SUBARCH'].strip().upper() == 'NO SHOW':
            df.at[index,'P1_SUBARCH'] = 'NO SHOW'
        if row['P2_SUBARCH'].strip().upper() == 'NO SHOW':
            df.at[index,'P2_SUBARCH'] = 'NO SHOW'

    # Format date column.
    df.EVENT_DATE = pd.to_datetime(df.EVENT_DATE,yearfirst=False,format='mixed')

    # Adding Event_IDs.
    count = 1000001
    df['EVENT_ID'] = 0
    for index, row in reversed(list(df.iterrows())):
        df.at[index,'EVENT_ID'] = count
        if pd.notna(row['EVENT_TYPE']):
            count += 1

    # Handle empty EVENT_TYPE/EVENT_DATE values by forward-filling.
    df['EVENT_TYPE'] = df['EVENT_TYPE'].ffill()
    df['EVENT_DATE'] = df['EVENT_DATE'].ffill()

    # Ignore events with incomplete data.
    events_to_ignore = [1000007]
    df = df[~df.EVENT_ID.isin(events_to_ignore)]

    # Replace Winner1/2 columns with single Match_Winner column.
    df['MATCH_WINNER'] = df.apply(lambda row: 'P1' if ((row['WINNER1'] == 1) & (row['WINNER2'] == 0)) else ('P2' if ((row['WINNER1'] == 0) & (row['WINNER2'] == 1)) else 'NA'), axis=1)
    df.drop(columns=['WINNER1','WINNER2'],inplace=True)

    # EVENT_ID 1000067 should be OTHER
    df.loc[df['EVENT_ID'] == 1000067,'EVENT_TYPE'] = 'OTHER'

    # Convert P1/P2_WINS from float to int.
    df['P1_WINS'] = df['P1_WINS'].astype(int)
    df['P2_WINS'] = df['P2_WINS'].astype(int)

    # Abstract out Event info into its own table.
    df_events = df.groupby(['EVENT_ID','EVENT_DATE']).agg({'EVENT_TYPE':'last'}).reset_index()

    return df, df_events

In [147]:
def merge_matches_codes(df_matches,df_valid_decks):
    df = pd.merge(left=df_matches,right=df_valid_decks,left_on=['P1_ARCH','P1_SUBARCH'],right_on=['ARCHETYPE','SUBARCHETYPE'],how='left')
    df.rename(columns={'DECK_ID':'P1_DECK_ID'}, inplace=True)
    df = pd.merge(left=df,right=df_valid_decks,left_on=['P2_ARCH','P2_SUBARCH'],right_on=['ARCHETYPE','SUBARCHETYPE'],how='left')
    df.rename(columns={'DECK_ID':'P2_DECK_ID'}, inplace=True)
    return df[['P1','P2','P1_WINS','P2_WINS','MATCH_WINNER','P1_DECK_ID','P2_DECK_ID','P1_NOTE','P2_NOTE','EVENT_ID']]

In [181]:
def merge_events_codes(df_events,df_valid_events):
    df = pd.merge(left=df_events,right=df_valid_events,left_on=['EVENT_TYPE'],right_on=['EVENT_TYPE'],how='left')
    return df[['EVENT_ID','EVENT_DATE','EVENT_TYPE_ID']]

In [199]:
def create_new_tables():
    create_valid_decks_query = """
    CREATE TABLE VALID_DECKS (
        FORMAT NVARCHAR(30),
        ARCHETYPE NVARCHAR(30),
        SUBARCHETYPE NVARCHAR(30),
        DECK_ID INT PRIMARY KEY
    );
    """
    create_valid_event_types_query = """
    CREATE TABLE VALID_EVENT_TYPES (
        FORMAT NVARCHAR(30),
        EVENT_TYPE NVARCHAR(30),
        EVENT_TYPE_ID INT PRIMARY KEY
    );
    """
    create_events_query = """
    CREATE TABLE EVENTS (
        EVENT_ID INT PRIMARY KEY,
        EVENT_DATE DATE,
        EVENT_TYPE_ID INT,
        FOREIGN KEY (EVENT_TYPE_ID) REFERENCES VALID_EVENT_TYPES(EVENT_TYPE_ID)
    );
    """
    create_matches_query = """
    CREATE TABLE MATCHES (
        MATCH_ID INT IDENTITY(1000000000,1) PRIMARY KEY,
        P1 NVARCHAR(30),
        P2 NVARCHAR(30),
        P1_WINS INT,
        P2_WINS INT,
        MATCH_WINNER NVARCHAR(2),
        P1_DECK_ID INT,
        P2_DECK_ID INT,
        P1_NOTE NVARCHAR(30),
        P2_NOTE NVARCHAR(30),
        EVENT_ID INT,
        FOREIGN KEY (P1_DECK_ID) REFERENCES VALID_DECKS(DECK_ID),
        FOREIGN KEY (P2_DECK_ID) REFERENCES VALID_DECKS(DECK_ID),
        FOREIGN KEY (EVENT_ID) REFERENCES EVENTS(EVENT_ID)
    );
    """
    try:
        conn(create_valid_decks_query)
        conn(create_valid_event_types_query)
        conn(create_events_query)
        conn(create_matches_query)
    except:
        print('Exception. No tables created.')

In [201]:
df_valid_decks, df_valid_event_types = parse_class_sheet(sheet_curr,gid_deck)
df_matches, df_events = parse_matchup_sheet(sheet_curr,gid_matches)
df_matches = merge_matches_codes(df_matches,df_valid_decks)
df_events = merge_events_codes(df_events,df_valid_event_types)

In [200]:
create_new_tables()

Exception. No tables created.


In [192]:
conn('DROP TABLE NewTable')

In [202]:
df_matches

,P1,P2,P1_WINS,P2_WINS,MATCH_WINNER,P1_DECK_ID,P2_DECK_ID,P1_NOTE,P2_NOTE,EVENT_ID
0,death_grips,etoustar,2,0,P1,1029,1001,NA,NA,1000075
1,etoustar,death_grips,0,2,P2,1001,1029,NA,NA,1000075
2,death_grips,_Chamytinho_,2,1,P1,1029,1029,NA,NA,1000075
3,etoustar,dingzhen,2,1,P1,1001,1031,NA,NA,1000075
4,_Chamytinho_,death_grips,1,2,P2,1029,1029,NA,NA,1000075
...,...,...,...,...,...,...,...,...,...,...
19039,2plus2isfive,Zenith777,1,2,P2,1027,1008,NA,NA,1000001
19040,s063,unluckymonkey,0,2,P2,1011,1022,NA,NA,1000001
19041,OrnatePuzzles,TonyScapone,1,2,P2,1031,1029,NA,NA,1000001
19042,Ale_Mtg,AFX,0,2,P2,1016,1007,NA,NA,1000001


In [203]:
df_valid_decks

,FORMAT,ARCHETYPE,SUBARCHETYPE,DECK_ID
0,VINTAGE,AGGRO,ELDRAZI,1001
1,VINTAGE,AGGRO,INITIATIVE,1002
2,VINTAGE,AGGRO,MERFOLK,1003
3,VINTAGE,AGGRO,OTHER AGGRO,1004
4,VINTAGE,AGGRO,RED PRISON,1005
5,VINTAGE,BAZAAR,AGGROVINE,1006
6,VINTAGE,BAZAAR,COUNTERVINE,1007
7,VINTAGE,BAZAAR,DREDGE,1008
8,VINTAGE,COMBO,BESEECH STORM,1009
9,VINTAGE,COMBO,BREACH,1010


In [204]:
df_valid_event_types

,FORMAT,EVENT_TYPE,EVENT_TYPE_ID
0,VINTAGE,CHALLENGE,101
1,VINTAGE,PRELIMINARY,102
2,VINTAGE,SHOWCASE,103
3,VINTAGE,SHOWCASE QUALIFIER,104
4,VINTAGE,OTHER,105


In [ ]:
df_events

,EVENT_ID,EVENT_DATE,EVENT_TYPE_ID
0,1000001,2024-08-29,101
1,1000002,2024-08-30,101
2,1000003,2024-08-31,101
3,1000004,2024-08-31,101
4,1000005,2024-09-01,101
...,...,...,...
69,1000071,2025-01-04,101
70,1000072,2025-01-05,101
71,1000073,2025-01-09,101
72,1000074,2025-01-10,101


: 

In [ ]:
df.to_excel('output.xlsx', index=False)